# Notebook 9: Prescriptive Final Validation

## 1. Introduction
This is the final notebook of the Prescriptive Analytics implementation. The goal is to move from a single-case test to a **systematic validation** of the optimizer's behavior.

We need to ensure that the engine is not just making lucky guesses, but is logically reacting to different production targets. We will test three primary scenarios:

1. **The Underproduction Scenario (Target $\gg$ Baseline)**: When the target is much higher than current capacity, does the AI recommend increasing resources?
2. **The Overproduction Scenario (Target $\ll$ Baseline)**: When the target is very low, does the AI recommend decreasing resources to save cost?
3. **The Stability Scenario (Target $\approx$ Baseline)**: When the target is already being met, does the AI suggest minimal changes?

If the optimizer passes these three tests, the prescriptive engine is considered validated and ready for deployment.

In [1]:
import pandas as pd
import numpy as np
import joblib
import itertools

# 1. Load Model
MODEL_PATH = '../models/production_model.joblib'
model = joblib.load(MODEL_PATH)

# 2. Re-integrate Optimizer Logic (from Notebook 8)
RESOURCE_CONSTRAINTS = {
    'no_of_workers': {'min': 8, 'max': 32, 'step': 1},
    'working_hours': {'min': 250.0, 'max': 1500.0, 'step': 8.0},
    'machine_count': {'min': 2, 'max': 7, 'step': 1}
}

def build_prescriptive_features(raw_inputs):
    features = raw_inputs.copy()
    workers = max(raw_inputs.get('no_of_workers', 1), 1)
    features['machine_worker_ratio'] = raw_inputs.get('machine_count', 0) / workers
    features['avg_hours_per_worker'] = raw_inputs.get('working_hours', 0) / workers
    defaults = {'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 'bottleneck_flag': False, 'rate': 5266.92}
    for col, value in defaults.items():
        if col not in features: features[col] = value
    return pd.DataFrame([features])

class PrescriptiveOptimizer:
    def __init__(self, model, constraints):
        self.model = model
        self.constraints = constraints
        self.w_gap = 1.0
        self.w_res = 0.1
    
    def generate_search_space(self):
        workers = range(self.constraints['no_of_workers']['min'], self.constraints['no_of_workers']['max'] + 1, self.constraints['no_of_workers']['step'])
        hours = np.arange(self.constraints['working_hours']['min'], self.constraints['working_hours']['max'] + 0.1, self.constraints['working_hours']['step'])
        machines = range(self.constraints['machine_count']['min'], self.constraints['machine_count']['max'] + 1, self.constraints['machine_count']['step'])
        return list(itertools.product(workers, hours, machines))
    
    def calculate_resource_cost(self, df):
        return ((df['no_of_workers']/32) + (df['working_hours']/1500) + (df['machine_count']/7)) / 3
    
    def optimize(self, target_qty, context_vars):
        search_space = self.generate_search_space()
        df_scenarios = pd.DataFrame(search_space, columns=['no_of_workers', 'working_hours', 'machine_count'])
        for col, val in context_vars.items(): df_scenarios[col] = val
        df_scenarios['machine_worker_ratio'] = df_scenarios['machine_count'] / df_scenarios['no_of_workers']
        df_scenarios['avg_hours_per_worker'] = df_scenarios['working_hours'] / df_scenarios['no_of_workers']
        defaults = {'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 'bottleneck_flag': False, 'rate': 5266.92}
        for col, value in defaults.items():
            if col not in df_scenarios.columns: df_scenarios[col] = value
        
        predictions = self.model.predict(df_scenarios)
        df_scenarios['predicted_qty'] = predictions
        df_scenarios['gap'] = abs(target_qty - df_scenarios['predicted_qty'])
        df_scenarios['cost'] = self.calculate_resource_cost(df_scenarios)
        df_scenarios['score'] = (self.w_gap * df_scenarios['gap']) + (self.w_res * df_scenarios['cost'])
        return df_scenarios.sort_values('score').head(1)

optimizer = PrescriptiveOptimizer(model, RESOURCE_CONSTRAINTS)
print("Optimization environment ready for validation.")

Optimization environment ready for validation.


## 2. Scenario-Based Validation

We will now run the optimizer against three distinct production targets to verify its behavior.

In [2]:
context = {
    'item_name': 'DIYANA - L - BROWN',
    'factory_unit': 'Factory Unit 01',
    'product_family': 'Gents',
    'size_id': 10,
    'grade': 'A',
    'order_category': 'Others'
}

scenarios = {
    'Underproduction (High Target)': 80, 
    'Overproduction (Low Target)': 20, 
    'Stability (Balanced Target)': 35
}

results = []

for name, target in scenarios.items():
    best_config = optimizer.optimize(target, context)
    res = best_config.iloc[0]
    results.append({
        'Scenario': name, 
        'Target': target, 
        'Workers': res['no_of_workers'], 
        'Hours': res['working_hours'], 
        'Machines': res['machine_count'], 
        'Predicted': res['predicted_qty'], 
        'Gap': res['gap']
    })

validation_df = pd.DataFrame(results)
display(validation_df)

,Scenario,Target,Workers,Hours,Machines,Predicted,Gap
0,Underproduction (High Target),80,8,1218.0,3,79.984835,0.015165
1,Overproduction (Low Target),20,12,410.0,2,19.996233,0.003767
2,Stability (Balanced Target),35,8,402.0,4,34.990088,0.009912


## 3. Final Verdict

To validate the engine, check the following:
1. **Scenario 1**: Are Workers/Hours/Machines higher than Scenario 2?
2. **Scenario 2**: Are Workers/Hours/Machines lower than Scenario 1?
3. **Scenario 3**: Is the Gap minimal?

If these patterns hold, the prescriptive engine is logically consistent and functionally correct.